In [ ]:
!pip install -q mlflow

In [ ]:
import os
import mlflow
from kaggle_secrets import UserSecretsClient

DATA_DIR = "/kaggle/input/competitions/ieee-fraud-detection"

secrets = UserSecretsClient()
os.environ["MLFLOW_TRACKING_USERNAME"] = "gabas22"
os.environ["MLFLOW_TRACKING_PASSWORD"] = secrets.get_secret("DAGSHUB_TOKEN")

mlflow.set_tracking_uri("https://dagshub.com/gabas22/ML-gabas22-Assignement-2.mlflow")
mlflow.set_experiment("LogisticRegression_Training")

In [ ]:
import pandas as pd

train_t = pd.read_csv(f"{DATA_DIR}/train_transaction.csv")
train_i = pd.read_csv(f"{DATA_DIR}/train_identity.csv")
test_t  = pd.read_csv(f"{DATA_DIR}/test_transaction.csv")
test_i  = pd.read_csv(f"{DATA_DIR}/test_identity.csv")

print("train_transaction:", train_t.shape)
print("train_identity:   ", train_i.shape)
print("test_transaction: ", test_t.shape)
print("test_identity:    ", test_i.shape)

In [ ]:
train = train_t.merge(train_i, how="left", on="TransactionID")
test  = test_t.merge(test_i,  how="left", on="TransactionID")

print("train merged:", train.shape)
print("test merged: ", test.shape)
print()
print("Fraud distribution:")
print(train["isFraud"].value_counts())
print("Fraud rate:", train["isFraud"].mean())

Cleaning

In [ ]:
nan_pct = train.isna().mean().sort_values(ascending=False)
print(nan_pct.head(20))
print()
print("columns with >90% missing:", (nan_pct > 0.9).sum())
print("columns with >50% missing:", (nan_pct > 0.5).sum())

In [ ]:
test = test.rename(columns={c: c.replace("-", "_") for c in test.columns})
print("test cols renamed")

In [ ]:
threshold = 0.8
drop_cols = nan_pct[nan_pct > threshold].index.tolist()
with mlflow.start_run(run_name="LogisticRegression_Cleaning_t08"):
    mlflow.log_param("nan_threshold", threshold)
    mlflow.log_param("dropped_cols", len(drop_cols))
    train_c = train.drop(columns=drop_cols)
    test_c  = test.drop(columns=drop_cols)
    
    mlflow.log_metric("cols_after", train_c.shape[1])
print("dropped:", len(drop_cols))
print("shape after:", train_c.shape)


In [ ]:
threshold = 0.7
drop_cols = nan_pct[nan_pct > threshold].index.tolist()
with mlflow.start_run(run_name="LogisticRegression_Cleaning_t07"):
    mlflow.log_param("nan_threshold", threshold)
    mlflow.log_param("dropped_cols", len(drop_cols))
    train_c = train.drop(columns=drop_cols)
    test_c  = test.drop(columns=drop_cols)
    
    mlflow.log_metric("cols_after", train_c.shape[1])
print("dropped:", len(drop_cols))
print("shape after:", train_c.shape)


Feature Engineering

In [ ]:
train_c["TransactionAmt"].describe()

In [ ]:
import numpy as np
train_c["TransactionAmt_log"] = np.log1p(train_c["TransactionAmt"])
test_c["TransactionAmt_log"] = np.log1p(test_c["TransactionAmt"])
train_c["hour"] = (train_c["TransactionDT"] // 3600) % 24
test_c["hour"] = (test_c["TransactionDT"] // 3600) % 24
train_c["day"] = (train_c["TransactionDT"] // (3600*24)) % 7
test_c["day"] = (test_c["TransactionDT"] // (3600*24)) % 7
print(train_c[["TransactionAmt", "TransactionAmt_log", "hour", "day"]].head())

In [ ]:
with mlflow.start_run(run_name="LogisticRegression_Feature_Engineering"):
    mlflow.log_param("log_transform", "TransactionAmt")
    mlflow.log_param("time_features", "hour,day")
    mlflow.log_metric("new_features", 3)
    mlflow.log_metric("cols_total", train_c.shape[1])
print("cols total:", train_c.shape[1])

Feature Selection

In [ ]:
threshold_corr = 0.95
num_cols = train_c.select_dtypes(include=[np.number]).columns.drop(["isFraud", "TransactionID"])
corr = train_c[num_cols].sample(100000, random_state=42).corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
drop_corr = [c for c in upper.columns if (upper[c] > threshold_corr).any()]
with mlflow.start_run(run_name="LogisticRegression_Feature_Selection"):
    mlflow.log_param("corr_threshold", threshold_corr)
    mlflow.log_param("dropped_corr", len(drop_corr))
    train_fs = train_c.drop(columns=drop_corr)
    test_fs = test_c.drop(columns=drop_corr)
    mlflow.log_metric("cols_after_fs", train_fs.shape[1])
print("dropped:", len(drop_corr))
print("shape after FS:", train_fs.shape)

Training

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
sample = train_fs.sample(200000, random_state=42)
y = sample["isFraud"]
X = sample.drop(columns=["isFraud", "TransactionID"]).copy()
for c in X.select_dtypes(include=["object"]).columns:
    X[c] = X[c].fillna("missing").astype("category").cat.codes
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
for C in [0.01, 1.0, 100]:
    pipe = Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler()), ("lr", LogisticRegression(max_iter=500, C=C))])
    scores = cross_val_score(pipe, X, y, cv=cv, scoring="roc_auc", n_jobs=-1)
    with mlflow.start_run(run_name=f"LogisticRegression_Training_C{C}"):
        mlflow.log_param("C", C)
        mlflow.log_param("max_iter", 500)
        mlflow.log_param("cv_folds", 3)
        mlflow.log_param("sample_size", len(X))
        mlflow.log_metric("cv_auc_mean", scores.mean())
        mlflow.log_metric("cv_auc_std", scores.std())
    print(f"C={C}: AUC={scores.mean():.4f} +/- {scores.std():.4f}")

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin
class FraudPrep(BaseEstimator, TransformerMixin):
    def __init__(self, drop_cols):
        self.drop_cols = drop_cols
    def fit(self, X, y=None):
        self.cat_maps_ = {}
        self.medians_ = {}
        Xt = self._prep(X.drop(columns=[c for c in self.drop_cols if c in X.columns]))
        for c in Xt.columns:
            if Xt[c].dtype == "object":
                cats = Xt[c].fillna("missing").astype(str).unique().tolist()
                self.cat_maps_[c] = {v: i for i, v in enumerate(cats)}
            else:
                self.medians_[c] = Xt[c].median()
        return self
    def transform(self, X):
        Xt = self._prep(X.drop(columns=[c for c in self.drop_cols if c in X.columns]))
        for c, m in self.cat_maps_.items():
            if c in Xt.columns:
                Xt[c] = Xt[c].fillna("missing").astype(str).map(m).fillna(-1).astype(int)
        for c, med in self.medians_.items():
            if c in Xt.columns:
                Xt[c] = Xt[c].fillna(med)
        return Xt.values
    def _prep(self, X):
        X = X.copy()
        X["TransactionAmt_log"] = np.log1p(X["TransactionAmt"])
        X["hour"] = (X["TransactionDT"] // 3600) % 24
        X["day"] = (X["TransactionDT"] // (3600*24)) % 7
        return X

In [ ]:
nan_pct = train.isna().mean()
nan_drop = nan_pct[nan_pct > 0.7].index.tolist()
all_drop = list(set(nan_drop) | set(drop_corr) | {"TransactionID"})
y_full = train["isFraud"]
X_full = train.drop(columns=["isFraud"])
final_pipe = Pipeline([("prep", FraudPrep(drop_cols=all_drop)), ("sc", StandardScaler()), ("lr", LogisticRegression(max_iter=500, C=1.0))])
final_pipe.fit(X_full, y_full)
print("fitted")

In [ ]:
import joblib
joblib.dump(final_pipe, "lr_pipeline.pkl")
test_preds = final_pipe.predict_proba(test)[:, 1]
print("preds shape:", test_preds.shape, "first:", test_preds[:3])
with mlflow.start_run(run_name="LogisticRegression_FinalPipeline"):
    mlflow.log_param("C", 1.0)
    mlflow.log_metric("cv_auc", 0.8282)
    mlflow.log_artifact("lr_pipeline.pkl")
print("saved")
